In [1]:
import heapq

n = 4

# Hướng di chuyển: xuống, trái, lên, phải
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

# Hàm chuyển ma trận thành tuple (để lưu visited)
def mat_to_tuple(mat):
    return tuple(num for row in mat for num in row)

# Heuristic: Manhattan Distance
def calculate_cost(mat, goal):
    cost = 0
    for i in range(n):
        for j in range(n):
            value = mat[i][j]
            if value != 0:
                # tìm vị trí đúng trong goal
                for x in range(n):
                    for y in range(n):
                        if goal[x][y] == value:
                            cost += abs(x - i) + abs(y - j)
    return cost

class Node:
    def __init__(self, parent, mat, empty_pos, level, cost):
        self.parent = parent
        self.mat = mat
        self.empty_pos = empty_pos
        self.level = level  # g(n)
        self.cost = cost    # h(n)

    def __lt__(self, other):
        return (self.cost + self.level) < (other.cost + other.level)

def new_node(mat, empty_pos, new_pos, level, parent, goal):
    new_mat = [row[:] for row in mat]

    x1, y1 = empty_pos
    x2, y2 = new_pos

    # swap
    new_mat[x1][y1], new_mat[x2][y2] = new_mat[x2][y2], new_mat[x1][y1]

    cost = calculate_cost(new_mat, goal)
    return Node(parent, new_mat, new_pos, level, cost)

def is_safe(x, y):
    return 0 <= x < n and 0 <= y < n

def print_matrix(mat):
    for row in mat:
        print(*row)
    print()

def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    print_matrix(node.mat)

def solve(initial, empty_pos, goal):
    pq = []
    visited = set()

    cost = calculate_cost(initial, goal)
    root = Node(None, initial, empty_pos, 0, cost)

    heapq.heappush(pq, root)

    while pq:
        current = heapq.heappop(pq)

        if current.cost == 0:
            print("=== TÌM THẤY LỜI GIẢI ===")
            print_path(current)
            return

        visited.add(mat_to_tuple(current.mat))

        for i in range(4):
            new_x = current.empty_pos[0] + rows[i]
            new_y = current.empty_pos[1] + cols[i]

            if is_safe(new_x, new_y):
                new_mat = [row[:] for row in current.mat]
                new_mat[current.empty_pos[0]][current.empty_pos[1]], new_mat[new_x][new_y] = \
                    new_mat[new_x][new_y], new_mat[current.empty_pos[0]][current.empty_pos[1]]

                if mat_to_tuple(new_mat) not in visited:
                    child = new_node(
                        current.mat,
                        current.empty_pos,
                        [new_x, new_y],
                        current.level + 1,
                        current,
                        goal
                    )
                    heapq.heappush(pq, child)

# ------------------ CHẠY ------------------
if __name__ == "__main__":
    initial = [
        [1, 2, 3, 4],
        [5, 6, 0, 8],
        [9,10, 7,11],
        [13,14,15,12]
    ]

    goal = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9,10,11,12],
        [13,14,15, 0]
    ]

    empty_pos = [1, 2]

    solve(initial, empty_pos, goal)

=== TÌM THẤY LỜI GIẢI ===
1 2 3 4
5 6 0 8
9 10 7 11
13 14 15 12

1 2 3 4
5 6 7 8
9 10 0 11
13 14 15 12

1 2 3 4
5 6 7 8
9 10 11 0
13 14 15 12

1 2 3 4
5 6 7 8
9 10 11 12
13 14 15 0

